In [7]:
import urllib.request
from pathlib import Path

import pandas as pd

In [9]:
def find_project_root(marker="data"):
    """Walk up from the current directory until a folder containing `marker` is found."""
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / marker).exists():
            return folder
    raise FileNotFoundError(f"No parent folder of {current} contains a '{marker}' directory")

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

BASE_ZA = "https://raw.githubusercontent.com/dsfsi/covid19za/master/data"
FILES = {
    "covid19za_provincial_confirmed.csv":   f"{BASE_ZA}/covid19za_provincial_cumulative_timeline_confirmed.csv",
    "covid19za_provincial_deaths.csv":      f"{BASE_ZA}/covid19za_provincial_cumulative_timeline_deaths.csv",
    "covid19za_provincial_vaccination.csv": f"{BASE_ZA}/covid19za_provincial_cumulative_timeline_vaccination.csv",
    "owid_vaccinations_full.csv": "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/vaccinations/vaccinations.csv",
}

OVERWRITE = False  # set True to refresh the data

print(f"Saving to: {RAW_DIR}\n")
for filename, url in FILES.items():
    path = RAW_DIR / filename
    if path.exists() and not OVERWRITE:
        print(f"Skipping {filename} (already exists)")
        continue
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, path)

# South Africa subset of the OWID file
owid_full = pd.read_csv(RAW_DIR / "owid_vaccinations_full.csv")
sa_vax = owid_full[owid_full["location"] == "South Africa"]
sa_vax.to_csv(RAW_DIR / "owid_vaccinations_south_africa.csv", index=False)
print(f"\nSouth Africa OWID rows: {len(sa_vax)}")

Saving to: C:\Users\shadr\hybrid-svir-ml\data\raw


South Africa OWID rows: 951
